In [9]:
# ===============================================================
# Extract Surface (0–10 m) BGC-Argo pCO2 + Temp + Salinity
# Keep only profiles where (platform_number, cycle_number) are DELAYED mode
# for TEMP, PSAL, and DOXY according to CMEMS index file.
#
# For CMEMS MULTIOBS BGC-ARGO files
# Louise Delaigue – 2025
# ===============================================================

import os
from tqdm import tqdm
import xarray as xr
import pandas as pd
import numpy as np


# ===============================================================
# Load CMEMS index file and build allowed (wmo, n_cycle) pairs
# ===============================================================
def load_allowed_pairs_from_index(index_path,
                                  require_temp=True,
                                  require_psal=True,
                                  require_doxy=True):
    """
    Reads the CMEMS index (CSV-like with '#' metadata lines) and returns:
      - allowed_pairs: set of (wmo(str), n_cycle(int)) where requested *_dt_mode == 'D'
      - idx: the index dataframe (for optional diagnostics)
    """
    idx = pd.read_csv(index_path, comment="#")

    required_cols = ["wmo", "n_cycle", "temp_dt_mode", "psal_dt_mode", "doxy_dt_mode"]
    missing = [c for c in required_cols if c not in idx.columns]
    if missing:
        raise ValueError(f"Index file missing columns: {missing}")

    idx["wmo"] = idx["wmo"].astype(str).str.strip()
    idx["n_cycle"] = idx["n_cycle"].astype(int)

    m = pd.Series(True, index=idx.index)
    if require_temp:
        m &= idx["temp_dt_mode"].astype(str).str.upper().str.strip().eq("D")
    if require_psal:
        m &= idx["psal_dt_mode"].astype(str).str.upper().str.strip().eq("D")
    if require_doxy:
        m &= idx["doxy_dt_mode"].astype(str).str.upper().str.strip().eq("D")

    allowed_pairs = set(zip(idx.loc[m, "wmo"], idx.loc[m, "n_cycle"]))
    return allowed_pairs, idx


# ===============================================================
# Extract per-profile surface means (0–10 m)
# ===============================================================
def extract_surface_means(nc_file, surface_depth=10):
    ds = xr.open_dataset(nc_file)

    # Required variables
    required = [
        "PRES_ADJUSTED",
        "PCO2_CONTENT", "PCO2_CONTENT_ERROR",
        "TEMP_ADJUSTED", "PSAL_ADJUSTED",
        "JULD", "LATITUDE", "LONGITUDE",
        "PLATFORM_NUMBER", "CYCLE_NUMBER"
    ]

    if any(v not in ds.variables for v in required):
        print(f"[WARNING] Required variable missing → {nc_file}")
        try:
            ds.close()
        except Exception:
            pass
        return pd.DataFrame()

    pres = ds["PRES_ADJUSTED"].values
    pco2 = ds["PCO2_CONTENT"].values
    pco2_err = ds["PCO2_CONTENT_ERROR"].values
    temp = ds["TEMP_ADJUSTED"].values
    psal = ds["PSAL_ADJUSTED"].values

    time = ds["JULD"].values
    lat = ds["LATITUDE"].values
    lon = ds["LONGITUDE"].values
    platform = ds["PLATFORM_NUMBER"].values
    cycle = ds["CYCLE_NUMBER"].values

    n_prof = ds.sizes["N_PROF"]
    results = []

    for i in range(n_prof):
        timestamp = pd.to_datetime(time[i])
        if pd.isna(timestamp):
            continue

        pres_i = pres[i, :]
        mask = (pres_i <= surface_depth) & (~np.isnan(pres_i))
        if not np.any(mask):
            continue

        co2 = pco2[i, mask]
        co2err = pco2_err[i, mask]
        t = temp[i, mask]
        s = psal[i, mask]

        if np.all(np.isnan(co2)):
            continue

        plat = platform[i].decode("utf-8").strip() if isinstance(platform[i], (bytes, np.bytes_)) else str(platform[i]).strip()
        cyc = int(cycle[i])

        results.append({
            "datetime": timestamp,
            "latitude": float(lat[i]),
            "longitude": float(lon[i]),
            "platform_number": plat,
            "cycle_number": cyc,
            "pco2": float(np.nanmean(co2)),
            "pco2_error": float(np.nanmean(co2err)),
            "temperature": float(np.nanmean(t)),
            "salinity": float(np.nanmean(s))
        })

    try:
        ds.close()
    except Exception:
        pass

    return pd.DataFrame(results)


# ===============================================================
# Paths
# ===============================================================
base_dir = "/remote/unity/bgc-output/DELAIGUE/SOCAT-FLOATS-OSSE/data/BGC-ARGO/cmems_obs-mob_glo_bgc-nut-car_mynrt_irr_i_202411"
index_path = "/remote/unity/bgc-output/DELAIGUE/SOCAT-FLOATS-OSSE/data/BGC-ARGO/canyon_index_file.txt"


# ===============================================================
# Build allowed delayed-mode (platform, cycle) set from index
# Requirement: TEMP, PSAL, DOXY are delayed mode ("D")
# ===============================================================
allowed_pairs, idx_df = load_allowed_pairs_from_index(
    index_path,
    require_temp=True,
    require_psal=True,
    require_doxy=True
)

print("\n====================================================")
print("Index filtering (delayed mode requirement)")
print("====================================================")
print(f"Allowed (platform, cycle) pairs where TEMP/PSAL/DOXY are 'D': {len(allowed_pairs)}")
print("====================================================")


# ===============================================================
# Main Loop
# ===============================================================
all_dfs = []
years = sorted([y for y in os.listdir(base_dir) if y.isdigit()])

for year in years:
    year_path = os.path.join(base_dir, year)
    if not os.path.isdir(year_path):
        continue

    print(f"\n=== YEAR {year} ===")

    for file in tqdm(sorted(os.listdir(year_path))):
        if not file.endswith(".nc"):
            continue

        fp = os.path.join(year_path, file)

        try:
            df = extract_surface_means(fp)
            if len(df) > 0:
                all_dfs.append(df)
        except Exception as e:
            print(f"[ERROR] {fp}: {e}")


# ===============================================================
# Combine extraction results (BEFORE delayed-mode filtering)
# ===============================================================
if len(all_dfs) > 0:
    final_df = pd.concat(all_dfs, ignore_index=True)
else:
    final_df = pd.DataFrame()

n_before = len(final_df)

print("\n====================================================")
print("Extraction complete (before delayed-mode filtering).")
print(f"Final dataset size: {n_before} profiles")
print("====================================================")


# ===============================================================
# Keep only (platform_number, cycle_number) present in delayed-mode index
# (TEMP/PSAL/DOXY delayed)
# ===============================================================
if len(final_df) > 0:
    final_df["platform_number"] = final_df["platform_number"].astype(str).str.strip()
    final_df["cycle_number"] = final_df["cycle_number"].astype(int)

    pair_tuples = list(zip(final_df["platform_number"], final_df["cycle_number"]))
    keep_mask = pd.Series(pair in allowed_pairs for pair in pair_tuples)

    final_df_delayed = final_df.loc[keep_mask].copy()
else:
    final_df_delayed = pd.DataFrame()

n_after = len(final_df_delayed)

print("\n====================================================")
print("Delayed-mode filtering stats (TEMP/PSAL/DOXY must be 'D')")
print("====================================================")
print(f"Profiles before filter: {n_before}")
print(f"Profiles after  filter: {n_after}")
print(f"Profiles removed:       {n_before - n_after}")
print("====================================================")


Index filtering (delayed mode requirement)
Allowed (platform, cycle) pairs where TEMP/PSAL/DOXY are 'D': 210445

=== YEAR 2002 ===


100%|██████████| 4/4 [00:00<00:00,  7.68it/s]



=== YEAR 2003 ===


100%|██████████| 12/12 [00:01<00:00,  8.33it/s]



=== YEAR 2004 ===


...


=== YEAR 2023 ===


100%|██████████| 12/12 [00:09<00:00,  1.26it/s]



=== YEAR 2024 ===


100%|██████████| 12/12 [00:08<00:00,  1.35it/s]


Extraction complete (before delayed-mode filtering).
Final dataset size: 230504 profiles

Delayed-mode filtering stats (TEMP/PSAL/DOXY must be 'D')
Profiles before filter: 230504
Profiles after  filter: 161206
Profiles removed:       69298


In [13]:
final_df[final_df.latitude <= -30]

,datetime,latitude,longitude,platform_number,cycle_number,pco2,pco2_error,temperature,salinity
13,2003-01-15 03:56:11.000146432,-47.889000,152.238000,5900245,1,329.411987,18.147499,12.284500,34.556499
15,2003-01-25 17:10:34.997168640,-47.262000,152.080000,5900245,2,314.049011,21.261501,12.133500,34.639000
17,2003-02-05 05:29:22.000030208,-47.248000,152.176000,5900245,3,326.791016,18.689499,12.541000,34.685001
20,2003-02-26 06:51:43.001110016,-47.117000,152.542000,5900245,5,327.988007,18.434000,12.476999,34.635498
24,2003-04-09 10:44:38.998374912,-46.332000,153.199000,5900245,9,324.366486,17.815500,14.898000,35.221001
...,...,...,...,...,...,...,...,...,...
230490,2024-12-31 14:32:34.001548288,-36.829000,96.429000,5907051,51,389.171997,31.247000,16.404636,35.029636
230492,2024-12-31 15:28:00.000000000,-55.525273,177.519958,5905556,81,377.618439,34.042427,8.766231,34.193928
230493,2024-12-31 16:02:12.001719296,-62.089700,-77.897100,5906553,72,403.709991,31.827999,4.538896,33.989262
230495,2024-12-31 16:24:07.001751552,-36.395100,-169.327400,5906567,76,416.127991,32.032001,20.671963,34.989758


In [14]:
final_df_delayed[final_df_delayed.latitude <= -30]

,datetime,latitude,longitude,platform_number,cycle_number,pco2,pco2_error,temperature,salinity
13,2003-01-15 03:56:11.000146432,-47.889000,152.238000,5900245,1,329.411987,18.147499,12.284500,34.556499
15,2003-01-25 17:10:34.997168640,-47.262000,152.080000,5900245,2,314.049011,21.261501,12.133500,34.639000
17,2003-02-05 05:29:22.000030208,-47.248000,152.176000,5900245,3,326.791016,18.689499,12.541000,34.685001
20,2003-02-26 06:51:43.001110016,-47.117000,152.542000,5900245,5,327.988007,18.434000,12.476999,34.635498
24,2003-04-09 10:44:38.998374912,-46.332000,153.199000,5900245,9,324.366486,17.815500,14.898000,35.221001
...,...,...,...,...,...,...,...,...,...
230490,2024-12-31 14:32:34.001548288,-36.829000,96.429000,5907051,51,389.171997,31.247000,16.404636,35.029636
230492,2024-12-31 15:28:00.000000000,-55.525273,177.519958,5905556,81,377.618439,34.042427,8.766231,34.193928
230493,2024-12-31 16:02:12.001719296,-62.089700,-77.897100,5906553,72,403.709991,31.827999,4.538896,33.989262
230495,2024-12-31 16:24:07.001751552,-36.395100,-169.327400,5906567,76,416.127991,32.032001,20.671963,34.989758


In [15]:
70448-62873

7575